# ERα MD: system preparation (reproducibility record)

This notebook rebuilds **every simulation input** for the MD part of manuscript 2026MP001112 from the files in
the `md-workflow` repository, using the **same pinned toolchain** that produced them:
`docker/tools-env.lock.txt` (explicit conda lock) + `gmx_MMPBSA 1.7.0`, `dimorphite-dl 2.1.0`.

| Step | Script | Output |
|---|---|---|
| Receptor: 5WGD → wild type + SRC2, loop graft, caps, His states | `prep/build_receptor.py` | `receptor/receptor_wt_src2.pdb`, `receptor_prep.yaml` |
| Ligands: SMILES + docked pose → stereo-checked, protonated ligand | `prep/check_ligands.py` (dry run) | printed table |
| Systems: GAFF2/AM1-BCC, tleap ff14SB/TIP3P, 0.15 M NaCl, GROMACS topologies, `grompp -maxwarn 0` | `prep/build_system.py` | `systems/<name>/provenance.yaml`, `systems/<name>/<pose>/build/` |
| Methods text from what was built | `prep/methods_text.py` | `docs/methods_text.md` |

**Runtime:** a standard Colab **CPU** runtime is enough (no GPU needed for preparation). The first run takes about
10 min for the environment and about 25 min for the 15 system builds. Production MD runs elsewhere (GPU image, see
`docs/MD_protocol.md`).

Everything is deterministic except the AM1-BCC charge optimisation (sqm), which reproduces to ~1e-4 e.

In [ ]:
# ---- parameters ------------------------------------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/eralpha-antidepressant-md"      # where the repo snapshot and outputs live on Google Drive
REPO_URL  = ""                                       # e.g. "https://github.com/<user>/eralpha-antidepressant-md.git" once pushed;
                                                     # empty -> use the snapshot DRIVE_DIR/md-workflow.tgz
WORK      = "/content/md-workflow"
ENV       = "/content/mamba/envs/tools"
SYSTEMS   = []                                       # [] = all systems in prep/systems.yaml
RUN_EM_TEST = True                                   # short CPU energy minimisation of one system as a sanity check

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(DRIVE_DIR, exist_ok=True); print(os.listdir(DRIVE_DIR))

## 1. Get the repository

In [ ]:
import os, subprocess
if os.path.exists(WORK):
    print("using existing", WORK)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
else:
    subprocess.run(["mkdir", "-p", WORK], check=True)
    subprocess.run(["tar", "-xzf", f"{DRIVE_DIR}/md-workflow.tgz", "-C", WORK], check=True)
print(sorted(os.listdir(WORK)))

## 2. Pinned toolchain (AmberTools 26, GROMACS 2026.3 CPU, RDKit, OpenMM/PDBFixer, PDB2PQR/PROPKA, gmx_MMPBSA)

In [ ]:
%%bash -s "$ENV" "$WORK"
set -euo pipefail
ENV=$1; WORK=$2
if [ ! -x "$ENV/bin/gmx" ]; then
  mkdir -p /content/bin && cd /content
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/2.3.2 | tar -xj bin/micromamba
  export MAMBA_ROOT_PREFIX=/content/mamba
  /content/bin/micromamba create -y -p "$ENV" --file "$WORK/docker/tools-env.lock.txt" > /content/env_create.log
  "$ENV/bin/pip" install -q "gmx_MMPBSA==1.7.0" "dimorphite-dl==2.1.0"
fi
echo "env ready: $(du -sh $ENV | cut -f1)"

In [ ]:
import os
os.environ["PATH"] = f"{ENV}/bin:" + os.environ["PATH"]
os.environ["AMBERHOME"] = ENV
os.chdir(WORK)
!gmx --version 2>/dev/null | grep -E "GROMACS version|Precision"
!python -I -c "import rdkit, openmm, parmed, pdbfixer, MDAnalysis; print('RDKit', rdkit.__version__, '| OpenMM', openmm.__version__, '| ParmEd', parmed.__version__, '| MDAnalysis', MDAnalysis.__version__)"
!pdb2pqr --version 2>&1 | tail -1; gmx_MMPBSA --version 2>&1 | tail -1

## 3. Receptor
5WGD chain A → S537Y reverted to wild type, SRC2 peptide kept, loop 461–472 grafted from SWISS-MODEL, ACE/NME caps,
PDB2PQR/PROPKA protonation at pH 7.4 with crystal estradiol present. Every decision is written to
`receptor/receptor_prep.yaml`.

In [ ]:
!python -I prep/build_receptor.py > /content/receptor.log 2>&1; tail -n 25 /content/receptor.log

## 4. Ligand dry run
For each ligand and pose: the intended microstate (charge, stereo from PubChem SMILES), how the docked pose was
mapped onto it, and, where the pose had the wrong configuration, the RMSD of the rebuilt stereoisomer to the pose.

In [ ]:
!python -I prep/check_ligands.py 2>&1 | grep -v "^\["

## 5. Build all systems (tleap → ParmEd → GROMACS, `grompp -maxwarn 0`)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-I", "prep/build_system.py", *SYSTEMS], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("==", "  "))))
if r.returncode: print(r.stderr[-4000:]); raise SystemExit("build failed")

In [ ]:
import yaml, glob, pandas as pd
rows = []
for f in sorted(glob.glob("systems/*/provenance.yaml")):
    pv = yaml.safe_load(open(f)); lig = pv.get("ligand", {})
    for pid, p in pv["poses"].items():
        s = p["system"]
        rows.append({"system": pv["system"], "pose": pid, "replicas": p["replicas"],
                     "charge": lig.get("formal_charge", ""), "CIP": lig.get("cip", ""),
                     "fit": p.get("pose", {}).get("method", "")[:40], "RMSD_A": p.get("pose", {}).get("heavy_rmsd_A", ""),
                     "atoms": s["atoms"], "waters": s["waters"], "Na+": s["Na+"], "Cl-": s["Cl-"],
                     "net_q": s["net_charge"], "box_A": s["box"]["a_A"],
                     "grompp_warn": p["grompp"]["em"]["warnings"] + p["grompp"]["nvt"]["warnings"]})
df = pd.DataFrame(rows); df

In [ ]:
assert (df.grompp_warn == 0).all() and (df.net_q.abs() < 1e-3).all(), "a system failed its checks"
print(len(df), "poses,", df.replicas.sum(), "replicas: all grompp-clean and neutral")

## 6. Methods text generated from what was built

In [ ]:
!python -I prep/methods_text.py
from IPython.display import Markdown; Markdown(open("docs/methods_text.md").read())

## 7. Optional sanity check: energy minimisation of one system on CPU (a few minutes)

In [ ]:
%%bash -s "$RUN_EM_TEST"
[ "$1" = "True" ] || exit 0
set -e
cd systems/estradiol/crystal/build && mkdir -p em_test && cd em_test
gmx grompp -f ../../../../../mdp/em.mdp -c ../conf.gro -r ../conf.gro -p ../topol.top -n ../index.ndx -o em.tpr -maxwarn 0 >/dev/null 2>&1
gmx mdrun -deffnm em -ntmpi 1 -ntomp $(nproc) 2>&1 | grep -E "converged|Potential Energy|Maximum force"

## 8. Save outputs to Google Drive

In [ ]:
%%bash -s "$DRIVE_DIR"
set -e
stamp=$(date +%Y%m%d-%H%M)
tar -czf "$1/prepared_systems_$stamp.tgz" receptor/receptor_wt_src2.pdb receptor/receptor_prep.yaml receptor/propka \
    systems/*/provenance.yaml systems/*/params systems/*/*/ligand.* systems/*/*/build docs/methods_text.md
ls -lh "$1" | tail -n 5